# Membership Inference
If you come from a security background, it may be easiest to view inference as a cross between fuzzing and cryptography that actually is easier than either on their own. Our goal will be to infer something we shouldn't know from an ML application. Normally this is the model itself, however the application it's used in determines the constraints and impacts. We will cover two of the three forms of inference. Black box inference, also referred to as blind inference, is typically involving a model hosted on a web page, and all you can do is provide input, and receive output from it. Grey box inference involves a model you don't have the training data for, but you do have access to the model itself. This could be a client side application, a model pilfered from a target, or an application with other vulnerabilities. We won't be covering White box entirely, because that implies there is nothing to infer, and everything is already known.

```
# This is formatted as code
```



## Blind Inference

Inference is often considered in the context of training data. Where the attack is inferring what training data was used to create a model. This comes in one of two forms. Reproducing the original training data, or inferring if a given piece of data is a member of the training set. In this first part we will be focused on reproducing training data used as is covered in the recent example of ["Extracting Training Data from Diffusion Models"](https://arxiv.org/abs/2301.13188). So first we will load the model they used in that paper.

In [ ]:
# DO NOT CHANGE

import random
import torch
from torch import nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

from mpl_toolkits.axes_grid1 import ImageGrid
from PIL import Image
from IPython import display
from matplotlib import pyplot as plt
from diffusers import StableDiffusionPipeline

from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else print("NoGPU"))

In [ ]:
# provided code
pipe = StableDiffusionPipeline.from_pretrained("compvis/stable-diffusion-v1-1", torch_dtype=torch.float16)
pipe = pipe.to(device)

The first step in real life would be generating 500 images, from every prompt however we have access to their paper, so we don't have to do quite as much work.

Exercise!

- Review the paper, ["Extracting Training Data from Diffusion Models"](https://arxiv.org/abs/2301.13188) and look for the information needed to finish the code bellow.
- Run it and see if you get results that match those of the paper.
- `i` is the seed to allow us all to get the same results


```{hint} prompt

- The prompt is on the first page of the paper...
```


:::

In [ ]:

prompt = "Ann Graham Lotz" # TODO: choose a prompt!

i = 716907
torch.manual_seed(i)
imagesnp = pipe(prompt, num_images_per_prompt=25, output_type="np.array").images
images = pipe.numpy_to_pil(imagesnp)

fig = plt.figure()
grid = ImageGrid(fig, 111,  # similar to subplot(111)
                 nrows_ncols=(5, 5),  # creates 2x2 grid of axes
                 )

for ax, im in zip(grid, images):
    ax.imshow(im)

plt.show()


Blind inference of this sort relies on the fact that an overfit training image will be strongly favored by the diffusion model and as such, multiple random seeds will settle onto the same pixel arrangement. As such, in this example it may be possible to visually identify the training image. With 500 images and a target that isn't as strongly favored, this may not be realistic however. So we can programmatically compare each image and look for ones that have a mean squared error less than a set value.

Exercise!

1. Rewrite the for loop to only show images that have an `MSELoss` less than a given value.
2. adjust that value to only show the images which are visually similar (L2 maybe?).

:::

In [ ]:
# provided code

lossfn = torch.nn.MSELoss()

imagest = torch.Tensor(imagesnp)
fig = plt.figure()
grid = ImageGrid(fig, 111,  # similar to subplot(111)
                 nrows_ncols=(5, 5),  # creates 2x2 grid of axes
                 )

for i in range(len(imagest)):
    # compare each image against the image 0 and image 23 as reference
    mse = lossfn(imagest[i], imagest[0]).item()

    # threshold controls strictness; lower = more similar
    if mse < 0.05:
        ax = grid[i]
        ax.imshow(images[i])
        ax.set_title(f"{mse:.4f}")
        ax.axis("off")
    else:
        grid[i].axis("off")

plt.show()

Exercise!

1. Change the prompt to "a gray sky" and observe how similar the images are.
2. Read section "4.2.1 Extraction Methodology" from ["Extracting Training Data from Diffusion Models"](https://arxiv.org/abs/2301.13188) and implement the solution they describe that doesn't involve finding clique size.


In [ ]:
# DO NOT CHANGE

prompt = "A gray sky"
i = 716907
torch.manual_seed(i)
imagesnp = pipe(prompt, num_images_per_prompt=25, output_type="np.array").images
images = pipe.numpy_to_pil(imagesnp)
imagest = torch.Tensor(imagesnp)
fig = plt.figure()
grid = ImageGrid(fig, 111,  # similar to subplot(111)
                 nrows_ncols=(5, 5),  # creates 2x2 grid of axes
                 )

for ax, im in zip(grid, images):
    # this part needs to be re-written
    ax.imshow(im)

plt.show()